# Models: smoke_test data → define → initialize → run

Self-contained, untrained teaching models. Each model has one code cell containing its definition, initialization, input, output, and readable table; no local project files are loaded. The standard XLM-R base configuration is downloaded/cached; weights are initialized randomly.

Five model objects: mention detection (XLM-R + BIO head), entity typing, description encoding, description matching, and entity disambiguation. Cached description vectors reduce runtime to four model objects.

## Complete flow: inputs → function → outputs

After initializing the models below, each call uses the previous steps' outputs:

```python
# 1. Detect mentions, decode their text/spans, and pool their vectors.
mention_vectors, mention_texts, mention_spans = mention_detection(**inputs, text=text, bio_labels=smoke_bio_labels)

# 2. Predict categories from mention context.
type_probabilities = entity_typing(mention_vectors)

# 3. Retrieve candidate entities and their stored evidence.
candidate_ids, candidate_records, known_types, priors = retriever(mention_texts)

# 4. Tokenize candidate names + descriptions.
description_texts = (candidate_records["label"] + " " + candidate_records["description"]).tolist()
description_inputs = description_tokenizer(description_texts, padding=True, truncation=True, max_length=32, return_tensors="pt")

# 5. Group tokens by mention and candidate, then encode descriptions.
description_ids = description_inputs["input_ids"].reshape(*priors.shape, -1)
description_mask = description_inputs["attention_mask"].reshape(*priors.shape, -1)
description_vectors = description_encoder(description_ids, description_mask)

# 6. Compare mention context with candidate descriptions.
description_probabilities = description_matching(mention_vectors, description_vectors, priors.ne(0))

# 7. Combine predicted types, priors, known types, and description matches.
_, entity_scores = entity_disambiguation(type_probabilities, priors, known_types, description_probabilities)

# 8. Select the highest-scoring column for each mention.
winning_columns = entity_scores.argmax(dim=1).tolist()
first_id = None if winning_columns[0] == priors.size(1) else candidate_ids[0][winning_columns[0]]
second_id = None if winning_columns[1] == priors.size(1) else candidate_ids[1][winning_columns[1]]
```

Final output: each mention's text, character span, and selected entity ID (`None` means no link). All rows keep the same mention order. `bio_labels` is optional; this smoke test supplies it instead of using the untrained classifier’s predictions.

In [1]:
import pandas as pd
import torch
from torch import nn
from transformers import XLMRobertaConfig, XLMRobertaModel
from IPython.display import display

torch.manual_seed(7)
torch.set_num_threads(2)

/mnt/storage/projects/learning-llm-components/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Mention detection: two mentions and their character spans

`mention_vectors, mention_texts, mention_spans = mention_detection(**inputs, text=text, bio_labels=smoke_bio_labels)`

Rows follow text order. Spans use start-inclusive/end-exclusive character positions. This no-loop smoke test supports exactly two mentions in one sequence; gold BIO labels bypass the untrained classifier for this smoke-test sentence: “Hawaii and Paris are places.”.

With `bio_labels`, use the supplied labels. Without it, use the classifier’s predictions. Both paths decode and pool in the same way; neither trains the model.

In [2]:
class MentionDetection(nn.Module):
    """Encode, detect, decode, and pool two mentions in one input sequence."""
    def __init__(self, config, num_labels=3):
        super().__init__()
        self.transformer = XLMRobertaModel(config, add_pooling_layer=False)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, token_ids, mask_ids, text, offsets, bio_labels=None):
        token_vectors = self.transformer(input_ids=token_ids, attention_mask=mask_ids).last_hidden_state
        if bio_labels is None:
            bio_labels = self.classifier(token_vectors).argmax(-1)
        else:
            # Supplied labels bypass prediction; this does not train the classifier.
            if bio_labels.shape != token_ids.shape[1:] or bio_labels.dtype != torch.long:
                raise ValueError("bio_labels must be a 1D integer tensor with one label per token.")
            if not ((bio_labels >= 0) & (bio_labels <= 2)).all():
                raise ValueError("BIO labels must be 0=O, 1=B, or 2=I.")
            bio_labels = bio_labels.to(token_ids.device).unsqueeze(0)
        membership, mention_texts, mention_spans = self.decode(bio_labels, mask_ids, text, offsets)
        weights = membership.to(token_vectors.dtype)
        mention_vectors = weights @ token_vectors[0] / weights.sum(dim=1, keepdim=True)
        return mention_vectors, mention_texts, mention_spans

    def decode(self, bio_labels, mask_ids, text, offsets):
        if bio_labels.size(0) != 1:
            raise ValueError("This smoke test expects one input sequence.")
        content = mask_ids[0].bool() & (offsets[:, 1] > offsets[:, 0])
        labels = bio_labels[0].masked_fill(~content, 0)
        starts = labels.eq(1).nonzero().flatten()
        if starts.numel() != 2:
            raise ValueError("This smoke test expects exactly two B/I mentions.")
        previous = torch.cat((labels.new_zeros(1), labels[:-1]))
        if (labels.eq(2) & previous.eq(0)).any():
            raise ValueError("An I label must follow B or I.")

        # One mask per mention; B starts a group and I continues it.
        group_ids = labels.eq(1).cumsum(0) - 1
        membership = group_ids.unsqueeze(0).eq(torch.arange(2, device=labels.device).unsqueeze(1)) & labels.ne(0)
        positions = torch.arange(labels.numel(), device=labels.device).expand(2, -1)
        ends = positions.masked_fill(~membership, -1).max(dim=1).values
        mention_spans = torch.stack((offsets[starts, 0], offsets[ends, 1]), dim=1).tolist()
        mention_texts = [text[mention_spans[0][0]:mention_spans[0][1]],
                         text[mention_spans[1][0]:mention_spans[1][1]]]
        return membership, mention_texts, mention_spans

# Standard XLM-R base configuration, unchanged; weights are randomly initialized.
xlmr_config = XLMRobertaConfig.from_pretrained("FacebookAI/xlm-roberta-base")
mention_config = {"num_labels": 3}
mention_detection = MentionDetection(xlmr_config, **mention_config).eval()

context_config = {
    "vocab_size": xlmr_config.vocab_size,
    "hidden_size": xlmr_config.hidden_size,
    "num_hidden_layers": xlmr_config.num_hidden_layers,
    "num_attention_heads": xlmr_config.num_attention_heads,
}

text = "Hawaii and Paris are places."
inputs = {
    "token_ids": torch.tensor([[0, 138393, 136, 7270, 621, 44677, 5, 2]]),
    "mask_ids": torch.tensor([[1, 1, 1, 1, 1, 1, 1, 1]]),
    "offsets": torch.tensor([[0, 0], [0, 6], [7, 10], [11, 16], [17, 20], [21, 27], [27, 28], [0, 0]]),
}
# Gold labels make this smoke test deterministic without training the model.
smoke_bio_labels = torch.tensor([0, 1, 0, 1, 0, 0, 0, 0])  # O, B, O, B, O, O, O, O.

mention_vectors, mention_texts, mention_spans = mention_detection(**inputs, text=text, bio_labels=smoke_bio_labels)
print("Mention vectors:", mention_vectors.shape)
mention_table = pd.DataFrame(mention_vectors.detach().numpy()).add_prefix("feature_")
mention_table.insert(0, "end", [mention_spans[0][1], mention_spans[1][1]])
mention_table.insert(0, "start", [mention_spans[0][0], mention_spans[1][0]])
mention_table.insert(0, "mention", mention_texts)
display(mention_table.iloc[:, :11])

/mnt/storage/projects/learning-llm-components/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Mention vectors: torch.Size([2, 768])


,mention,start,end,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7
0,Hawaii,0,6,-0.185084,-0.110405,1.274982,-0.865854,-0.519592,0.070957,-0.560580,1.515596
1,Paris,11,16,-1.623274,-0.992972,1.128416,-1.362223,-1.039562,0.541162,0.073226,0.581863


## 2. Entity typing: one probability per type

Predict categories from the mention vectors. The shared `type_names` order also defines the KB type-vector columns used by the next step.

In [3]:
# Shared type vocabulary
type_names = ["place", "state", "island"]

class EntityTyping(nn.Module):
    """Mention vectors -> independent probabilities for each type."""
    def __init__(self, num_types, hidden_size=8):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, num_types)

    def forward(self, mention_vectors):
        return self.classifier(mention_vectors).sigmoid()

# Initialize
typing_config = {"num_types": len(type_names), "hidden_size": mention_vectors.shape[-1]}
entity_typing = EntityTyping(**typing_config).eval()

type_probabilities = entity_typing(mention_vectors)

type_table = pd.DataFrame(type_probabilities.detach().numpy(), columns=type_names)
type_table.insert(0, "mention", mention_texts)
display(type_table)

,mention,place,state,island
0,Hawaii,0.327072,0.518739,0.559801
1,Paris,0.337698,0.387276,0.624254


## 3. Candidate retrieval: input and output

Initialize `CandidateRetriever` with `kb`, `type_names`, and `num_candidates`. Call `retriever(mention_texts)` to retrieve candidates.

| Input / setting | Example / meaning |
|---|---|
| `mention_texts` | `["Hawaii", "Paris"]` |
| `kb` | Prepared table of mentions, entities, descriptions, priors, and types |
| `type_names` | `["place", "state", "island"]`; defines vector column order |
| `num_candidates` | `1`; return the highest-prior candidate per mention (use `2` for both) |

| Output | Shape in this example | Meaning |
|---|---|---|
| `candidate_ids` | 2 × K nested list | Entity IDs grouped by mention |
| `candidate_records` | 2 × K table rows | Retrieved KB fields plus `mention_index` |
| `known_types` | `[2, K, 3]` tensor | Mention × candidate × known type membership |
| `priors` | `[2, K]` tensor | Mention × candidate lookup preference |

This retriever uses exact name matching. An alternative retriever can keep the same inputs, outputs, and ordering.

`K = num_candidates`. Candidates are ranked by prior within each mention. Priors retain their original KB values; they are not renormalized after selection. Requesting more candidates than the KB contains raises a clear error.

In [4]:
# Mock retrieval inputs
# Each row is a possible entity for a stored name/alias.
# label/description identify the entity; type columns are known 1/0 memberships.
# Priors are invented preferences, not measurements or model predictions.
kb = pd.DataFrame({
    "mention": ["Hawaii", "Hawaii", "Paris", "Paris"],
    "entity_id": ["Q782", "Q68740", "Q90", "Q830149"],
    "label": ["Hawaii", "Hawaii", "Paris", "Paris"],
    "description": ["state of the United States of America", "largest of the Hawaiian islands",
                    "capital of France", "city in Texas"],
    "prior": [0.85, 0.15, 0.90, 0.10],
    "place": [1., 1., 1., 1.],
    "state": [1., 0., 0., 0.],
    "island": [0., 1., 0., 0.],
})
display(kb)

,mention,entity_id,label,description,prior,place,state,island
0,Hawaii,Q782,Hawaii,state of the United States of America,0.85,1.0,1.0,0.0
1,Hawaii,Q68740,Hawaii,largest of the Hawaiian islands,0.15,1.0,0.0,1.0
2,Paris,Q90,Paris,capital of France,0.90,1.0,0.0,0.0
3,Paris,Q830149,Paris,city in Texas,0.10,1.0,0.0,0.0


In [5]:
class CandidateRetriever:
    """Retrieve the highest-prior candidates for each mention, preserving mention order."""
    def __init__(self, kb: pd.DataFrame, type_names: list[str], num_candidates: int = 2):
        if not isinstance(num_candidates, int) or isinstance(num_candidates, bool) or num_candidates < 1:
            raise ValueError("num_candidates must be a positive integer.")
        self.kb = kb
        self.type_names = type_names
        self.num_candidates = num_candidates

    def __call__(self, mention_texts: list[str]) -> tuple[
        list[list[str]], pd.DataFrame, torch.Tensor, torch.Tensor
    ]:
        """Return IDs [M,C], records [M*C rows], known types [M,C,T], priors [M,C]."""
        num_mentions = len(mention_texts)
        if num_mentions == 0:
            raise ValueError("Provide at least one mention for this retrieval demo.")

        mention_rows = pd.DataFrame({"mention_index": range(num_mentions), "mention": mention_texts})
        records = mention_rows.merge(self.kb, on="mention", how="left", sort=False)
        if records["entity_id"].isna().any():
            raise ValueError("A mention has no candidates in the mock KB.")
        if records.groupby("mention_index").size().min() < self.num_candidates:
            raise ValueError("A mention has fewer KB candidates than requested; reduce num_candidates.")

        # Select top candidates per mention, rather than requiring an exact KB row count.
        records = records.sort_values(["mention_index", "prior"], ascending=[True, False], kind="stable")
        records = records.groupby("mention_index", sort=False).head(self.num_candidates).reset_index(drop=True)

        candidate_ids = records["entity_id"].to_numpy().reshape(num_mentions, self.num_candidates).tolist()
        known_types = torch.tensor(records[self.type_names].to_numpy(), dtype=torch.float32)
        known_types = known_types.reshape(num_mentions, self.num_candidates, len(self.type_names))
        priors = torch.tensor(records["prior"].to_numpy(), dtype=torch.float32)
        priors = priors.reshape(num_mentions, self.num_candidates)
        return candidate_ids, records, known_types, priors


# Initialize once; only mention texts are passed at retrieval time.
retriever = CandidateRetriever(kb=kb, type_names=type_names, num_candidates=2)
candidate_ids, candidate_records, known_types, priors = retriever(mention_texts)
num_mentions, num_candidates = priors.shape

candidate_table = candidate_records.copy()
display(candidate_table)

,mention_index,mention,entity_id,label,description,prior,place,state,island
0,0,Hawaii,Q782,Hawaii,state of the United States of America,0.85,1.0,1.0,0.0
1,0,Hawaii,Q68740,Hawaii,largest of the Hawaiian islands,0.15,1.0,0.0,1.0
2,1,Paris,Q90,Paris,capital of France,0.90,1.0,0.0,0.0
3,1,Paris,Q830149,Paris,city in Texas,0.10,1.0,0.0,0.0


In [6]:
known_types

tensor([[[1., 1., 0.],
         [1., 0., 1.]],

        [[1., 0., 0.],
         [1., 0., 0.]]])

## 4. Description encoder: text becomes candidate vectors

Tokenize each retrieved entity's name + description with the standard XLM-R tokenizer, then encode it. Padding and attention masks are created automatically.

In [7]:
selected_ids = candidate_records["entity_id"]
selected_ids

0       Q782
1     Q68740
2        Q90
3    Q830149
Name: entity_id, dtype: object

In [8]:
from transformers import AutoTokenizer

# Use the tokenizer matching the encoder's vocabulary.
description_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base", use_fast=True)
description_texts = (candidate_records["label"] + " " + candidate_records["description"]).tolist()

# The tokenizer adds special tokens, pads descriptions, and creates attention masks.
description_inputs = description_tokenizer(
    description_texts,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors="pt",
)

# Group the flat candidate rows into [mentions, candidates, tokens].
input_shape = (num_mentions, num_candidates, -1)
description_ids = description_inputs["input_ids"].reshape(input_shape)
description_mask = description_inputs["attention_mask"].reshape(input_shape)

/mnt/storage/projects/learning-llm-components/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# Define: token embeddings → masked average → description vector.
class DescriptionEncoder(nn.Module):
    """Input [mentions, candidates, tokens]; output [mentions, candidates, features]."""
    def __init__(self, vocabulary_size, hidden_size=8, description_size=4):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, hidden_size, padding_idx=1)
        self.projection = nn.Linear(hidden_size, description_size)

    def forward(self, description_ids, description_mask):
        token_vectors = self.embedding(description_ids)
        token_mask = description_mask.bool().unsqueeze(-1)

        # Sum real-token vectors, then divide by their count.
        summed_vectors = (token_vectors * token_mask).sum(dim=-2)
        token_counts = token_mask.sum(dim=-2).clamp_min(1)
        mean_vectors = summed_vectors / token_counts
        return self.projection(mean_vectors)


# Initialize.
description_config = {
    "vocabulary_size": xlmr_config.vocab_size,
    "hidden_size": 8,
    "description_size": 4,
}
description_encoder = DescriptionEncoder(**description_config).eval()

# Run: one output vector per retrieved candidate.
description_vectors = description_encoder(description_ids, description_mask)
print("Description vectors:", description_vectors.shape)

# Display one row per candidate alongside its readable description.
description_table = candidate_records[["mention", "entity_id"]].reset_index(drop=True)
description_table["description_input"] = (
    candidate_records["label"] + " " + candidate_records["description"]
).to_numpy()
vector_rows = description_vectors.detach().flatten(0, 1).numpy()
vector_table = pd.DataFrame(vector_rows).add_prefix("feature_")
description_table = pd.concat([description_table, vector_table], axis=1)
display(description_table)

Description vectors: torch.Size([2, 2, 4])


,mention,entity_id,description_input,feature_0,feature_1,feature_2,feature_3
0,Hawaii,Q782,Hawaii state of the United States of America,0.120816,-0.071838,-0.160232,0.514286
1,Hawaii,Q68740,Hawaii largest of the Hawaiian islands,-0.065374,-0.017283,-0.289155,0.354884
2,Paris,Q90,Paris capital of France,0.367552,0.008401,0.093268,0.678512
3,Paris,Q830149,Paris city in Texas,-0.100595,-0.066707,-0.207383,-0.124540


In [10]:
description_vectors

tensor([[[ 0.1208, -0.0718, -0.1602,  0.5143],
         [-0.0654, -0.0173, -0.2892,  0.3549]],

        [[ 0.3676,  0.0084,  0.0933,  0.6785],
         [-0.1006, -0.0667, -0.2074, -0.1245]]], grad_fn=<ViewBackward0>)

## 5. Description matching: one probability per candidate, plus none

In [11]:
class DescriptionMatching(nn.Module):
    """Project mention vectors; compare with candidate description vectors."""
    def __init__(self, hidden_size=8, description_size=4):
        super().__init__()
        self.mention_projection = nn.Linear(hidden_size, description_size)

    def forward(self, mention_vectors, description_vectors, candidate_mask):
        projected = self.mention_projection(mention_vectors)
        similarities = (description_vectors * projected.unsqueeze(1)).sum(-1)
        similarities = similarities.masked_fill(~candidate_mask, -1e8)
        none_score = similarities.new_zeros((similarities.size(0), 1))
        return torch.cat((similarities, none_score), dim=1).softmax(-1)

# Initialize
matching_config = {"hidden_size": mention_vectors.shape[-1], "description_size": description_vectors.shape[-1]}
description_matching = DescriptionMatching(**matching_config).eval()

description_probabilities = description_matching(mention_vectors, description_vectors, priors.ne(0))

score_entity_ids = candidate_ids[0] + ["NONE"] + candidate_ids[1] + ["NONE"]
score_mentions = [mention_texts[0]] * (num_candidates + 1) + [mention_texts[1]] * (num_candidates + 1)
match_table = pd.DataFrame({"mention": score_mentions, "entity_id": score_entity_ids,
                            "description_match": description_probabilities.detach().flatten().tolist()})
display(match_table)

,mention,entity_id,description_match
0,Hawaii,Q782,0.401367
1,Hawaii,Q68740,0.306018
2,Hawaii,NONE,0.292615
3,Paris,Q90,0.356736
4,Paris,Q830149,0.334088
5,Paris,NONE,0.309175


## 6. Entity disambiguation: compare evidence and score candidates

Stored KB types and priors meet predicted types and description matches here. Each candidate supplies six features; the model returns one score per candidate plus NONE.

In [12]:
class EntityDisambiguation(nn.Module):
    """Same scoring features and inference interface as the upstream ED layer."""
    def __init__(self, num_types):
        super().__init__()
        self.classifier = nn.Linear(num_types + 3, 1)

    def candidate_features(self, class_activations, candidate_pem_values,
                           candidate_classes, candidate_description_scores):
        predicted_types = class_activations.unsqueeze(1)
        type_agreement = candidate_classes * predicted_types
        type_distance = torch.linalg.vector_norm(candidate_classes - predicted_types, dim=-1, keepdim=True)
        return torch.cat((type_agreement, candidate_pem_values.unsqueeze(-1),
                          type_distance, candidate_description_scores[:, :-1].unsqueeze(-1)), dim=-1)

    def forward(self, class_activations, candidate_pem_values, candidate_classes,
                candidate_description_scores, current_device='cpu'):
        features = self.candidate_features(class_activations, candidate_pem_values,
                                           candidate_classes, candidate_description_scores)
        scores = self.classifier(features.to(current_device)).squeeze(-1)
        scores = scores.masked_fill(candidate_pem_values.to(current_device).eq(0), -1e8)
        none_score = scores.new_zeros((scores.size(0), 1))
        return None, torch.cat((scores, none_score), dim=1)

# Initialize
disambiguation_config = {"num_types": len(type_names)}
entity_disambiguation = EntityDisambiguation(**disambiguation_config).eval()

# Inspect input evidence
features = entity_disambiguation.candidate_features(type_probabilities, priors, known_types, description_probabilities)

evidence_table = pd.DataFrame(features.detach().reshape(-1, features.size(-1)).numpy(), columns=["place_agreement", "state_agreement", "island_agreement", "prior", "type_distance", "description_match"])
evidence_table.insert(0, "entity_id", candidate_records["entity_id"].tolist())
evidence_table.insert(0, "mention", candidate_records["mention"].tolist())
display(evidence_table)

_, entity_scores = entity_disambiguation(
    class_activations=type_probabilities,
    candidate_pem_values=priors,
    candidate_classes=known_types,
    candidate_description_scores=description_probabilities,
    current_device="cpu",
)
assert entity_scores.shape == (num_mentions, num_candidates + 1)

,mention,entity_id,place_agreement,state_agreement,island_agreement,prior,type_distance,description_match
0,Hawaii,Q782,0.327072,0.518739,0.000000,0.85,0.998910,0.401367
1,Hawaii,Q68740,0.327072,0.000000,0.559801,0.15,0.956921,0.306018
2,Paris,Q90,0.337698,0.000000,0.000000,0.90,0.989100,0.356736
3,Paris,Q830149,0.337698,0.000000,0.000000,0.10,0.989100,0.334088


## 7. Select the highest score; untrained scores have no factual meaning

In [13]:
winning_columns = entity_scores.argmax(dim=1).tolist()
first_id = None if winning_columns[0] == num_candidates else candidate_ids[0][winning_columns[0]]
second_id = None if winning_columns[1] == num_candidates else candidate_ids[1][winning_columns[1]]
results = pd.DataFrame({
    "mention": mention_texts,
    "start": [mention_spans[0][0], mention_spans[1][0]],
    "end": [mention_spans[0][1], mention_spans[1][1]],
    "entity_id": [first_id, second_id],
    "untrained_demo": True,
})
score_table = pd.DataFrame({"mention": score_mentions, "entity_id": score_entity_ids,
                            "score": entity_scores.detach().flatten().tolist()})
score_table["selected"] = torch.arange(num_candidates + 1).unsqueeze(0).eq(entity_scores.argmax(dim=1, keepdim=True)).flatten().tolist()
display(score_table)

,mention,entity_id,score,selected
0,Hawaii,Q782,-0.200600,False
1,Hawaii,Q68740,-0.140251,False
2,Hawaii,NONE,0.000000,True
3,Paris,Q90,-0.029740,False
4,Paris,Q830149,0.050764,True
5,Paris,NONE,0.000000,False


In [14]:
display(results)

,mention,start,end,entity_id,untrained_demo
0,Hawaii,0,6,None,True
1,Paris,11,16,Q830149,True


## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| Untrained weights | Scores do not establish the correct entity | Use the pretrained pipeline for real linking |
| Gold BIO labels | The demo checks decoding and pooling, not detection accuracy | Omit `bio_labels` to use classifier predictions after training |
| Fixed mock inputs | Token IDs and candidate data only describe the Hawaii/Paris example | Use the production tokenizer and KB |
| Simplified description encoder | Uses mean embeddings instead of ReFinED's transformer | Use the pretrained description encoder or cached vectors |
